Import Libraries

In [2]:
from typing import TypedDict, List, Optional
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent
from langchain_tavily import TavilySearch
from pydantic import BaseModel, Field
import chromadb
import gradio as gr
import os
from dotenv import load_dotenv

load_dotenv()
print("Imports Completed!!!")

Imports Completed!!!


Initialize Claude & Tavily Search 

In [3]:
llm = ChatAnthropic (
    model = "claude-sonnet-4-6",
    temperature = 0.3
)

search_tool = TavilySearch(
    max_results = 3,
    topic="general"
)

print("Claude and Tavily Ready!!!")

Claude and Tavily Ready!!!


Define the Agent State

In [4]:
class AgentState(TypedDict):
    dietary_needs: str
    meal_plan: str
    health_critique: str
    search_results: List[str]
    ingredients: List[str]
    inventory_matches: List[str]
    cart: List[str]
    approved: bool
    iterations: int

print("Agent State Defined and Ready!!!")

Agent State Defined and Ready!!!


Set up ChromaDB with grocery inventory

In [5]:
# Initialize ChromaDB
chroma_client = chromadb.Client()

# Create grocery inventory collection
inventory_collection = chroma_client.create_collection(
    name="grocery_inventory"
)

# Grocery store inventory data
grocery_items = [
    # Proteins
    "chicken breast - protein, lean meat, 165 calories per 100g",
    "salmon fillet - protein, omega-3, healthy fats, 208 calories per 100g",
    "tofu firm - plant protein, vegan, low calorie, 76 calories per 100g",
    "eggs large - protein, vitamin D, 155 calories per 100g",
    "lentils - plant protein, fiber, iron, 116 calories per 100g",
    "chickpeas - plant protein, fiber, vegan, 164 calories per 100g",
    "greek yogurt - protein, probiotics, calcium, 59 calories per 100g",
    # Vegetables
    "spinach fresh - iron, vitamins, low calorie, 23 calories per 100g",
    "broccoli - vitamin C, fiber, anti-inflammatory, 34 calories per 100g",
    "sweet potato - complex carbs, vitamin A, fiber, 86 calories per 100g",
    "avocado - healthy fats, potassium, fiber, 160 calories per 100g",
    "bell peppers - vitamin C, antioxidants, low calorie, 31 calories per 100g",
    "kale - superfood, vitamins K C A, calcium, 49 calories per 100g",
    "carrots - beta carotene, fiber, vitamin A, 41 calories per 100g",
    # Grains
    "quinoa - complete protein, gluten free, fiber, 120 calories per 100g",
    "brown rice - complex carbs, fiber, gluten free, 216 calories per 100g",
    "oats rolled - fiber, beta glucan, heart healthy, 389 calories per 100g",
    "whole wheat bread - fiber, complex carbs, 247 calories per 100g",
    # Fruits
    "blueberries - antioxidants, vitamin C, low sugar, 57 calories per 100g",
    "banana - potassium, energy, natural sugar, 89 calories per 100g",
    "apple - fiber, vitamin C, antioxidants, 52 calories per 100g",
    "lemon - vitamin C, detox, low calorie, 29 calories per 100g",
    # Healthy fats
    "olive oil - healthy fats, anti-inflammatory, 884 calories per 100g",
    "almonds - healthy fats, protein, vitamin E, 579 calories per 100g",
    "chia seeds - omega-3, fiber, plant protein, 486 calories per 100g",
    # Dairy alternatives
    "almond milk - low calorie, vegan, calcium fortified, 17 calories per 100g",
    "coconut milk - healthy fats, vegan, creamy, 230 calories per 100g",
]

# Add items to ChromaDB with IDs
inventory_collection.add(
    documents=grocery_items,
    ids=[f"item_{i}" for i in range(len(grocery_items))]
)

print(f"Grocery inventory loaded: {len(grocery_items)} items")
print("ChromaDB vector store ready!")

Grocery inventory loaded: 27 items
ChromaDB vector store ready!


The Grocery Search Tool

In [6]:
@tool
def search_grocery_inventory(query: str) -> str:
    """Search the grocery store inventory for items matching dietary needs.
    Input should be a natural language query like 'high protein vegetarian' 
    or 'low calorie vegetables'."""
    try:
        results = inventory_collection.query(
            query_texts=[query],
            n_results=5
        )
        items = results["documents"][0]
        return f"Found {len(items)} matching items:\n" + "\n".join([f"- {item}" for item in items])
    except Exception as e:
        return f"Error: {str(e)}"

print("Grocery search tool ready!")

Grocery search tool ready!


Define the AgentState nodes.

In [7]:
def chef_agent(state: AgentState) -> dict:
    print("\n👨‍🍳 Chef Agent thinking...")
    response = llm.invoke([
        SystemMessage(content="""You are a top class home cooking chef. 
        Your cooking style matches mom's cooking which is very personal and emotional to an individual. 
        You have to play a role of a cooking assistant and help pick healthy ingredients to make an awesome
        meal that's quick, easy and tasty.
        Always list the specific ingredients needed at the end of your response in a clear list."""),
        HumanMessage(content=f"Dietary needs and preferences: {state['dietary_needs']}")
    ])
    print("✅ Meal plan created")
    return {
        "meal_plan": response.content,
        "iterations": state.get("iterations", 0) + 1
    }

def dr_ganapathy_agent(state: AgentState) -> dict:
    print("\n👨‍⚕️ Dr. Ganapathy reviewing...")
    response = llm.invoke([
        SystemMessage(content="""You are Dr. Advice Ganapathy, an experienced nutritionist 
        and health advisor. Review the proposed meal plan and ingredients critically:
        1. Check if ingredients align with the patient's dietary needs
        2. Evaluate overall health quotient of the meal
        3. Consider appropriateness for time of day
        4. Flag any health concerns or contraindications
        5. Suggest healthier alternatives where needed
        If the meal plan is nutritionally sound, start with 'APPROVED'.
        Otherwise provide specific improvements needed."""),
        HumanMessage(content=f"""
Patient dietary needs: {state['dietary_needs']}
Proposed meal plan: {state['meal_plan']}
Please review and provide your medical nutrition assessment.""")
    ])
    print("✅ Health review complete")
    print(f"   Approved: {'APPROVED' in response.content}")
    return {"health_critique": response.content}

def research_agent(state: AgentState) -> dict:
    print("\n🔍 Research Agent searching...")
    search_results = []
    
    # Search for dietary recommendations
    queries = [
        f"{state['dietary_needs']} dietary recommendations",
        f"nutritional benefits {state['meal_plan'][:50]}",
        "healthy meal planning guidelines"
    ]
    
    for query in queries:
        try:
            results = search_tool.invoke(query)
            if isinstance(results, list):
                for r in results:
                    if isinstance(r, dict):
                        search_results.append(r.get("content", ""))
            elif isinstance(results, str):
                search_results.append(results)
        except Exception as e:
            print(f"   Search error: {e}")
    
    print(f"✅ Found {len(search_results)} research items")
    return {"search_results": search_results}

def grocery_matcher_agent(state: AgentState) -> dict:
    print("\n🛒 Grocery Matcher searching inventory...")
    
    # Extract ingredients from meal plan
    ingredient_response = llm.invoke([
        SystemMessage(content="""Extract only the ingredient names from the meal plan. 
        Return a simple comma separated list of ingredients only. 
        No quantities, no instructions, just ingredient names."""),
        HumanMessage(content=state["meal_plan"])
    ])
    
    ingredients = [i.strip() for i in ingredient_response.content.split(",")]
    print(f"   Ingredients identified: {len(ingredients)}")
    
    # Search ChromaDB for each ingredient
    inventory_matches = []
    for ingredient in ingredients[:8]:  # limit to 8 ingredients
        results = search_grocery_inventory.invoke(ingredient)
        inventory_matches.append(f"{ingredient}: {results.split(chr(10))[1] if chr(10) in results else results}")
    
    # Build shopping cart
    cart = [f"✅ {ingredient}" for ingredient in ingredients[:8]]
    
    print(f"✅ Cart ready with {len(cart)} items")
    return {
        "ingredients": ingredients,
        "inventory_matches": inventory_matches,
        "cart": cart
    }

print("All agent nodes defined!")

All agent nodes defined!


Conditional Edges and Supervisor Logic

In [8]:
def should_continue(state: AgentState) -> str:
    iterations = state.get("iterations", 0)
    health_critique = state.get("health_critique", "")

    print(f"\n Checking - iterations: {iterations}")
    
    if "APPROVED" in health_critique:
        print("→ Dr. Ganapathy approved - moving to research")
        return "research"
    if iterations >= 3:
        print("→ Max iterations reached - moving to research")
        return "research"
        
    print("→ Needs improvement - back to Chef")
    return "chef"
    
print("Condition defined")

Condition defined


Build the Graph

In [9]:
workflow = StateGraph(AgentState)

#All Nodes
workflow.add_node("chef_agent", chef_agent)
workflow.add_node("dr_ganapathy_agent", dr_ganapathy_agent)
workflow.add_node("research_agent", research_agent)
workflow.add_node("grocery_matcher_agent", grocery_matcher_agent)

#Fixed Edges
workflow.add_edge("chef_agent", "dr_ganapathy_agent")
workflow.add_edge("research_agent", "grocery_matcher_agent")

# Conditional edge after Dr Ganapathy reviews
workflow.add_conditional_edges("dr_ganapathy_agent", should_continue, {
    "chef" : "chef_agent",
    "research" : "research_agent",
})

workflow.set_entry_point("chef_agent")
print("Graph is Built & ready!!!")

Graph is Built & ready!!!


Compile and Invoke

In [10]:
app = workflow.compile()
result = app.invoke({
    # "dietary_needs": "Looking for a high protein vegan meal for the afternoon",
    # "dietary_needs": "Indian Breakfast meal similar to Dosa or Idly for a power start in the morning",
    "dietary_needs": "Solid dinner needed for a 12 year old female tennis player for a big day in the tennis court tomorrow morning",
    "meal_plan": "",
    "health_critique": "",
    "search_results": [],
    "ingredients": [],
    "inventory_matches": [],
    "cart": [],
    "approved": False,
    "iterations": 0
})

print(f"\n✅ Completed in {result['iterations']} iterations")
print(f"📚 Research items: {len(result['search_results'])} items")
print("\n👨‍🍳 Final Meal Plan:")
print("=" * 60)
print(result["meal_plan"])
print("\n👨‍⚕️ Dr. Ganapathy's Review:")
print("=" * 60)
print(result["health_critique"])
print("\n🛒 Shopping Cart:")
print("=" * 60)
for item in result["cart"]:
    print(item)


👨‍🍳 Chef Agent thinking...
✅ Meal plan created

👨‍⚕️ Dr. Ganapathy reviewing...
✅ Health review complete
   Approved: True

 Checking - iterations: 1
→ Dr. Ganapathy approved - moving to research

🔍 Research Agent searching...
✅ Found 0 research items

🛒 Grocery Matcher searching inventory...
   Ingredients identified: 20
✅ Cart ready with 8 items

✅ Completed in 1 iterations
📚 Research items: 0 items

👨‍🍳 Final Meal Plan:
# 🎾 Power-Up Dinner for Your Tennis Star!

What a fun challenge! For a 12-year-old tennis player with a big match tomorrow, we want a meal that:

- **Loads up on complex carbs** for sustained energy
- **Has good protein** for muscle support
- **Is easy to digest** overnight
- **Tastes absolutely delicious** - because she's 12 and it has to be yummy! 😄

---

## 🍝 Mom's Champion Pasta Bowl

**Whole wheat pasta with a hearty turkey tomato sauce and a simple side salad**

This is the perfect pre-competition meal! Think of it like a warm hug in a bowl - comforting, filli

Gradio Integration

In [11]:
import gradio as gr

def run_grocery_agent(dietary_input: str) -> tuple:
    """Run the grocery agent and return formatted results."""
    
    if not dietary_input.strip():
        return "Please enter your dietary needs!", "", ""
    
    print(f"\n🚀 Starting agent for: {dietary_input}")
    
    try:
        result = app.invoke({
            "dietary_needs": dietary_input,
            "meal_plan": "",
            "health_critique": "",
            "search_results": [],
            "ingredients": [],
            "inventory_matches": [],
            "cart": [],
            "approved": False,
            "iterations": 0
        })
        
        # Format meal plan
        meal_plan = result.get("meal_plan", "No meal plan generated")
        
        # Format health critique
        health_critique = result.get("health_critique", "No review available")
        
        # Format shopping cart
        cart_items = result.get("cart", [])
        cart_text = "\n".join(cart_items) if cart_items else "No items in cart"
        
        return meal_plan, health_critique, cart_text
        
    except Exception as e:
        return f"Error: {str(e)}", "", ""

# Build Gradio interface
with gr.Blocks(
    title="Personal Grocery & Meal Agent",
    theme=gr.themes.Soft()
) as demo:
    
    gr.Markdown("""
    # 🥗 Personal Grocery & Meal Planning Agent
    ### Powered by Chef AI + Dr. Ganapathy Health Review
    *Tell me your dietary needs and I'll plan your perfect meal and shopping list!*
    """)
    
    with gr.Row():
        with gr.Column():
            dietary_input = gr.Textbox(
                label="🍽️ What would you like to cook today?",
                placeholder="e.g. I want a high protein vegan lunch, or quick vegetarian dinner for 2, or healthy breakfast with eggs...",
                lines=3
            )
            submit_btn = gr.Button(
                "Get My Meal Plan! 🚀",
                variant="primary",
                size="lg"
            )
    
    gr.Markdown("---")
    
    with gr.Row():
        with gr.Column():
            meal_output = gr.Markdown(
                label="👨‍🍳 Chef's Meal Plan"
            )
    
    with gr.Row():
        with gr.Column():
            health_output = gr.Markdown(
                label="👨‍⚕️ Dr. Ganapathy's Health Review"
            )
    
    with gr.Row():
        with gr.Column():
            cart_output = gr.Textbox(
                label="🛒 Your Shopping Cart",
                lines=10,
                interactive=False
            )
    
    # Example queries
    gr.Examples(
        examples=[
            ["High protein vegan meal for afternoon energy"],
            ["Quick vegetarian dinner ready in 20 minutes"],
            ["Healthy breakfast with eggs, plant based sides"],
            ["Low calorie lunch for weight loss"],
            ["Post workout meal with high protein"],
        ],
        inputs=dietary_input,
        label="💡 Try these examples"
    )
    
    # Connect button to function
    submit_btn.click(
        fn=run_grocery_agent,
        inputs=dietary_input,
        outputs=[meal_output, health_output, cart_output]
    )

print("Launching Personal Grocery Agent...")
demo.launch(share=True)

C:\Users\tenni\AppData\Local\Temp\ipykernel_11896\3466429715.py:40: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Launching Personal Grocery Agent...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://1433c178e1300994dc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


C:\Users\tenni\ai-agents-learning\venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)



🚀 Starting agent for: Tasty Cracked Wheat Upma

👨‍🍳 Chef Agent thinking...
✅ Meal plan created

👨‍⚕️ Dr. Ganapathy reviewing...
✅ Health review complete
   Approved: True

 Checking - iterations: 1
→ Dr. Ganapathy approved - moving to research

🔍 Research Agent searching...
✅ Found 0 research items

🛒 Grocery Matcher searching inventory...
   Ingredients identified: 15
✅ Cart ready with 8 items


C:\Users\tenni\ai-agents-learning\venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)



🚀 Starting agent for: Vegetarian Mint rice with protein and carbs for the day, have soy chunks 7 can have eggs on the side for lunch tomorrow.

👨‍🍳 Chef Agent thinking...
✅ Meal plan created

👨‍⚕️ Dr. Ganapathy reviewing...
✅ Health review complete
   Approved: True

 Checking - iterations: 1
→ Dr. Ganapathy approved - moving to research

🔍 Research Agent searching...
✅ Found 0 research items

🛒 Grocery Matcher searching inventory...
   Ingredients identified: 23
✅ Cart ready with 8 items
